In [2]:
import json
import tiktoken

In [3]:
infra_path = "../../infra/"
json_path = f"{infra_path}json/"
cuad_json = f"{infra_path}CUAD_v1/CUAD_v1.json"
main_document = "BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement"
cuad_json, json_path

('../../infra/CUAD_v1/CUAD_v1.json', '../../infra/json/')

In [4]:
with open(cuad_json, "r") as f:
    cuad_data = json.load(f)

In [5]:
len(cuad_data['data'])

510

In [6]:
from pydantic import BaseModel, Field
from typing import List, Optional
from pathlib import Path
import re

class Node(BaseModel):
    id: str
    text: str
    relationsCount: int = 0

class Edge(BaseModel):
    source: str
    target: str
    type: str
    score: Optional[float] = None
    ref_label: Optional[str] = None
    ref_value: Optional[str] = None

class Graph(BaseModel):
    nodes: List[Node]
    edges: List[Edge]

class Config:
  CUAD_PDF_DIR = Path("../infra/CUAD_v1/full_contract_pdf")
  CUAD_DOC_DIR = Path("../infra/CUAD_v1/full_contract_docx")
  EDITED_PDF_DIR = Path("../infra/edited_pdfs")
  EDITED_DOC_DIR = Path("../infra/edited_docs")
  REFERENCE_PATTERNS = [
    ("section", re.compile(r'\b[Ss]ection\s+(\d+(?:\.\d+)*)\b')),
    ("article", re.compile(r'\b[Aa]rticle\s+(\d+(?:\.\d+)*)\b')),
    ("schedule", re.compile(r'\b[Ss]chedule\s+([A-Za-z]|\d+(?:\.\d+)*)\b')),
    ("annex",    re.compile(r'\b[Aa]nnex\s+([A-Za-z]|\d+(?:\.\d+)*)\b')),
    ("appendix", re.compile(r'\b[Aa]ppendix\s+([A-Za-z]|\d+(?:\.\d+)*)\b')),
  ]
    
  TYPE_PRIORITY = {
    "precedence": 5,
    "scope": 4,
    "deontic": 3,
    "numeric": 2,
    "definition": 2,
    "other": 1
  }

#### Extract context

In [7]:
new_json_only_context = []

COST_INPUT_PER_MILLION = 0.40  # millon tokens
COST_OUTPUT_PER_MILLION = 1.60  # millon tokens
cost_input_model_per_token = (COST_INPUT_PER_MILLION / 1_000_000)
cost_output_model_per_token = (COST_OUTPUT_PER_MILLION / 1_000_000)
enc = tiktoken.encoding_for_model("gpt-4.1-mini")

index = 1
for item in cuad_data['data']:
    title = item['title']
    context = item['paragraphs'][0]['context']
    number_tokens = len(enc.encode(context))

    new_json_only_context.append({
        "id": f"CUAD_{index}",
        "title": title,
        "context": context,
        'total_number_tokens': number_tokens,
        'cost_document_input': ((number_tokens / 1_000_000) * COST_INPUT_PER_MILLION),
        'cost_document_output': ((number_tokens / 1_000_000) * COST_OUTPUT_PER_MILLION)
    })

    index += 1

In [8]:
document_output_path = f"{json_path}rerank/context.json" 

with open(document_output_path, "w") as f:
    json.dump(new_json_only_context, f, indent=4)

In [9]:
INPUT_PATH = f"{json_path}/rerank/context.json"
OUTPUT_PATH = f"{json_path}/rerank/paragraphs.json"

MAX_DOCS = 1

def split_into_paragraphs(text: str):
    text = text.replace("\r\n", "\n").strip()
    parts = re.split(r"\n\s*\n+", text)

    out = []
    for part in parts:
        clean = " ".join(part.split())
        if clean:
            out.append(clean)

    if out:
        return out

    clean = " ".join(text.split())
    return [clean] if clean else []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    docs = json.load(f)

candidate_docs = docs

print(len(candidate_docs), "\n")

selected = []
for i, doc in enumerate(candidate_docs, start=1):
    title = doc.get("title", "")
    context = doc.get("context", "")
    total_tokens_doc = len(enc.encode(context))

    if title == main_document:
        print(title)
        selected.append(doc)
        break 

selected = sorted(selected, key=lambda d: len(enc.encode(d.get("context", ""))), reverse=False)

print(len(selected), "\n")

stage1 = []

for doc in selected[:MAX_DOCS]:
    paragraphs = []

    for i, p_text in enumerate(split_into_paragraphs(doc.get("context", "")), start=1):
        n_tokens = len(enc.encode(p_text))

        paragraphs.append(
            {
                "idx": f"IDX{i}",
                "text": p_text,
                "number_tokens": n_tokens,
                "cost_document_input": (n_tokens / 1_000_000) * cost_input_model_per_token,
                "cost_document_output": (n_tokens / 1_000_000) * cost_output_model_per_token,
            }
        )

    stage1.append(
        {
            "doc_id": doc.get("id"),
            "num_paragraphs": len(paragraphs),
            "total_number_tokens": sum(p["number_tokens"] for p in paragraphs),
            "cost_document_input": sum(p["cost_document_input"] for p in paragraphs),
            "paragraphs": paragraphs,
        }
    )

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(stage1, f, indent=2, ensure_ascii=False)

510 

BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement
1 



In [10]:
from collections import defaultdict, Counter

INPUT_PATH = f"{json_path}/rerank/context.json"
OUTPUT_PATH = f"{json_path}/rerank/words.json"

def tokenize(text: str):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

def split_into_paragraphs(text: str):
    text = text.replace("\r\n", "\n").strip()
    parts = re.split(r"\n\s*\n+", text)

    out = []
    for part in parts:
        clean = " ".join(part.split())
        if clean:
            out.append(clean)

    if out:
        return out

    clean = " ".join(text.split())
    return [clean] if clean else []

def map_reduce(docs):
    mapped = []

    # -------- MAP --------
    for doc in docs:
        doc_id = doc.get("id")

        for p_text in split_into_paragraphs(doc.get("context", "")):
            words = tokenize(p_text)

            for w in words:
                mapped.append((doc_id, w))

    # -------- REDUCE --------
    reduced = defaultdict(list)

    for doc_id, word in mapped:
        reduced[doc_id].append(word)

    final_output = []

    for doc_id, words in reduced.items():
        counter = Counter(words)

        sorted_words = dict(
            sorted(counter.items(), key=lambda x: x[1], reverse=True)
        )

        final_output.append({
            "doc_id": doc_id,
            "words": sorted_words
        })

    return final_output

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    docs = json.load(f)

index = map_reduce(docs)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(index, f, indent=2, ensure_ascii=False)

In [11]:
from collections import defaultdict, Counter

INPUT_PATH = f"{json_path}/rerank/context.json"
OUTPUT_PATH = f"{json_path}/rerank/words_paragraph.json"

def tokenize(text: str):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

def split_into_paragraphs(text: str):
    text = text.replace("\r\n", "\n").strip()
    parts = re.split(r"\n\s*\n+", text)

    out = []
    for part in parts:
        clean = " ".join(part.split())
        if clean:
            out.append(clean)

    if out:
        return out

    clean = " ".join(text.split())
    return [clean] if clean else []

def map_reduce(docs):
    final_output = []

    for doc in docs:
        doc_id = doc.get("id")
        paragraphs_output = []

        for i, p_text in enumerate(split_into_paragraphs(doc.get("context", "")), start=1):
            paragraph_id = f"IDX{i}"

            words = tokenize(p_text)
            counter = Counter(words)

            sorted_words = dict(
                sorted(counter.items(), key=lambda x: x[1], reverse=True)
            )

            paragraphs_output.append({
                "idx": paragraph_id,
                # "text": p_text,
                "words": sorted_words
            })

        final_output.append({
            "doc_id": doc_id,
            "paragraphs": paragraphs_output
        })

    return final_output

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    docs = json.load(f)

index = map_reduce(docs)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(index, f, indent=2, ensure_ascii=False)

In [12]:
from sentence_transformers import SentenceTransformer, util

def generate_graph_data(paragraphs_data: list) -> Graph:
    model = SentenceTransformer('all-MiniLM-L6-v2')
    nodes: List[Node] = []
    edges: List[Edge] = []
    
    for p in paragraphs_data:
        node = Node(
            id=str(p.get("idx")),
            text=p.get("text", "").strip(),
            relationsCount=0,
        )
        nodes.append(node)
    
    for i in range(len(nodes)):
        current_text = nodes[i].text
        for ref_type, pattern in Config.REFERENCE_PATTERNS:
            matches = pattern.finditer(current_text)
            for match in matches:
                ref_id = match.group(1)
                for target_node in nodes:
                    if target_node.id != nodes[i].id and target_node.text.startswith(ref_id):
                        edges.append(Edge(
                            source=nodes[i].id, 
                            target=target_node.id, 
                            type="reference",
                            ref_label=ref_type,
                            ref_value=ref_id
                        ))

    if nodes:
        embeddings = model.encode([n.text for n in nodes], convert_to_tensor=True)
        cosine_scores = util.cos_sim(embeddings, embeddings)

        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                score = float(cosine_scores[i][j])
                if score > 0.8:
                    edges.append(Edge(
                        source=nodes[i].id, 
                        target=nodes[j].id, 
                        type="semantic_similarity", 
                        score=score
                    ))

    relations_map = {}
    for edge in edges:
        relations_map[edge.source] = relations_map.get(edge.source, 0) + 1
        relations_map[edge.target] = relations_map.get(edge.target, 0) + 1
    
    for node in nodes:
        node.relationsCount = relations_map.get(node.id, 0)

    return Graph(nodes=nodes, edges=edges)

/home/luis/miniconda3/envs/doc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
graph = generate_graph_data(stage1[0]["paragraphs"])
graph

/home/luis/miniconda3/envs/doc/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 803: system has unsupported display driver / cuda driver combination (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Graph(nodes=[Node(id='IDX1', text='Exhibit 10.1 [***] = Certain confidential information contained in this document, marked by brackets, has been omitted because it is both (i) not material and (ii) would likely be competitively harmful if publicly disclosed.', relationsCount=0), Node(id='IDX2', text='Miltenyi Biotec-Bellicum Supply Agreement (Execution Copy March 27, 2019)', relationsCount=53), Node(id='IDX3', text='SUPPLY AGREEMENT', relationsCount=0), Node(id='IDX4', text='(MB Global Contract Number MBGCR 19001)', relationsCount=0), Node(id='IDX5', text='This Supply Agreement (this "Agreement") is made and entered into, effective as of March 27, 2019 (the "Effective Date"), by and between Miltenyi Biotec GmbH, a German corporation having an address at Friedrich-Ebert-Str. 68, 51429 Bergisch Gladbach, Germany (hereinafter referred to as "Miltenyi"), and Bellicum Pharmaceuticals, Inc., a US corporation, having a registered office at 2130 West Holcombe Boulevard, Suite 800, Houston, TX